# UK Inflation Forecasting – Merging New Bank Rate Data

In this notebook, we will:

1. Load the **clean merged monthly dataset** (`clean_merged_dataset.csv`)
2. Load the **daily Bank Rate data** from the Bank of England (`BoE-Database_export (2).csv`)
3. Convert the daily Bank Rate series to **monthly frequency**
4. Merge the monthly Bank Rate into the cleaned dataset (replacing the old Bank Rate column)
5. Save an updated final dataset for modelling (e.g. LSTM)


In [1]:
import pandas as pd

print("Pandas version:", pd.__version__)

# === 1) Load cleaned merged monthly dataset ===
merged = pd.read_csv("clean_merged_dataset.csv")
print("Merged dataset shape:", merged.shape)
print("Merged dataset columns:")
print(merged.columns.tolist())

# === 2) Load Bank Rate daily dataset ===
boe = pd.read_csv("BoE-Database_export (2).csv")  # change name if your file is different
print("\nBoE Bank Rate dataset shape:", boe.shape)
print("BoE dataset columns:")
print(boe.columns.tolist())

print("\n=== Merged dataset (head) ===")
print(merged.head())

print("\n=== BoE Bank Rate dataset (head) ===")
print(boe.head())


Pandas version: 2.3.3
Merged dataset shape: (338, 8)
Merged dataset columns:
['CPI ANNUAL RATE 00: ALL ITEMS 2015=100', 'Date', 'YearMonth', 'Gross Value Added - Monthly (3 month on 3 month growth) :CVM SA', 'Bank Rate', 'Exchange_USD', 'Unemployment rate (aged 16 and over, seasonally adjusted): %', 'RPI: Percentage change over 12 months - Petrol and Oil incl Fuel Oil']

BoE Bank Rate dataset shape: (7309, 2)
BoE dataset columns:
['Date', 'Value']

=== Merged dataset (head) ===
   CPI ANNUAL RATE 00: ALL ITEMS 2015=100        Date YearMonth  \
0                                     1.7  1997-06-01   1997-06   
1                                     2.0  1997-07-01   1997-07   
2                                     2.0  1997-08-01   1997-08   
3                                     1.8  1997-09-01   1997-09   
4                                     1.9  1997-10-01   1997-10   

   Gross Value Added - Monthly (3 month on 3 month growth) :CVM SA  Bank Rate  \
0                                

## Step – Convert Date to Datetime and Keep Only One Observation per Month

Since the dataset contains multiple observations within the same month, we will:

1. Convert the `Date` column to proper `datetime` format  
2. Create a `YearMonth` column (YYYY-MM)  
3. Group by `YearMonth` and select the **earliest date** within each month  
4. Keep only one row per month for a clean monthly time series  

This ensures the dataset is consistent with monthly macroeconomic data standards.


In [3]:
# 1. Convert Date column to datetime
boe['Date'] = pd.to_datetime(boe['Date'], errors='coerce')

# 2. Create YearMonth column (YYYY-MM)
boe['YearMonth'] = boe['Date'].dt.to_period('M').astype(str)

# 3. Extract earliest observation per month
df_monthly = (
    boe.sort_values('Date')
      .groupby('YearMonth', as_index=False)
      .first()
)

# 4. Convert YearMonth back to a proper datetime (use month start)
df_monthly['Date'] = pd.to_datetime(df_monthly['YearMonth'])

# Display result
print("Shape after monthly aggregation:", df_monthly.shape)
df_monthly.head()


Shape after monthly aggregation: (349, 3)


,YearMonth,Date,Value
0,1997-01,1997-01-01,5.9375
1,1997-02,1997-02-01,5.9375
2,1997-03,1997-03-01,5.9375
3,1997-04,1997-04-01,5.9375
4,1997-05,1997-05-01,5.9375


## Step – Inspect Column Names and Date Ranges Before Merging

Before merging the clean merged dataset and the monthly Bank Rate dataset, we
must confirm:

1. Column names in both datasets
2. That both contain consistent `Date` and `YearMonth` fields
3. The start and end dates for both datasets
4. Their overlapping time range for correct merging

This prevents alignment errors and ensures a clean, accurate merge.


In [5]:

# === 1. Check column names ===
print("Main dataset columns:")
print(merged.columns.tolist())

print("\nBank Rate dataset columns:")
print(df_monthly.columns.tolist())


# === 2. Ensure Date columns are datetime ===
merged['Date'] = pd.to_datetime(merged['Date'], errors='coerce')
df_monthly['Date'] = pd.to_datetime(df_monthly['Date'], errors='coerce')

# === 3. Check Date range for main dataset ===
print("\nMain dataset date range:")
print("Start:", merged['Date'].min(), "  End:", merged['Date'].max())

# === 4. Check Date range for Bank Rate monthly dataset ===
print("\nBank Rate dataset date range:")
print("Start:", df_monthly['Date'].min(), "  End:", df_monthly['Date'].max())

# === 5. Check YearMonth alignment ===
print("\nMain YearMonth sample:", merged['YearMonth'].head())
print("Bank YearMonth sample:", df_monthly['YearMonth'].head())


Main dataset columns:
['CPI ANNUAL RATE 00: ALL ITEMS 2015=100', 'Date', 'YearMonth', 'Gross Value Added - Monthly (3 month on 3 month growth) :CVM SA', 'Bank Rate', 'Exchange_USD', 'Unemployment rate (aged 16 and over, seasonally adjusted): %', 'RPI: Percentage change over 12 months - Petrol and Oil incl Fuel Oil']

Bank Rate dataset columns:
['YearMonth', 'Date', 'Value']

Main dataset date range:
Start: 1997-06-01 00:00:00   End: 2025-07-01 00:00:00

Bank Rate dataset date range:
Start: 1997-01-01 00:00:00   End: 2025-12-01 00:00:00

Main YearMonth sample: 0    1997-06
1    1997-07
2    1997-08
3    1997-09
4    1997-10
Name: YearMonth, dtype: object
Bank YearMonth sample: 0    1997-01
1    1997-02
2    1997-03
3    1997-04
4    1997-05
Name: YearMonth, dtype: object


## Step – Merge Monthly Bank Rate Into Main Dataset Using YearMonth

We have:

- `merged` → main macro dataset (1997-06 to 2025-07), already monthly  
- `df_monthly` → monthly Bank Rate (1997-01 to 2025-12), with columns:
  - `YearMonth`
  - `Date`
  - `Value` (Bank Rate)

In this step we:
1. Drop the old (mostly empty) `Bank Rate` column from `merged`
2. Rename `Value` → `Bank Rate` in `df_monthly`
3. Merge `Bank Rate` into `merged` using `YearMonth`
4. Inspect the result.


In [6]:

# Just to be safe: ensure YearMonth is string in both
merged['YearMonth'] = merged['YearMonth'].astype(str)
df_monthly['YearMonth'] = df_monthly['YearMonth'].astype(str)

# 1. Drop old Bank Rate column from merged (we'll replace it)
if 'Bank Rate' in merged.columns:
    merged = merged.drop(columns=['Bank Rate'])

# 2. Prepare Bank Rate dataset: rename Value -> Bank Rate
bank_monthly = df_monthly.rename(columns={'Value': 'Bank Rate'})

# 3. Merge on YearMonth (left join: keep all rows from merged)
merged_with_rate = merged.merge(
    bank_monthly[['YearMonth', 'Bank Rate']],  # only need these columns
    on='YearMonth',
    how='left'
)

print("Shape before merge:", merged.shape)
print("Shape after merge :", merged_with_rate.shape)

# 4. Quick check
merged_with_rate[['Date', 'YearMonth', 'Bank Rate']].head()


Shape before merge: (338, 7)
Shape after merge : (338, 8)


,Date,YearMonth,Bank Rate
0,1997-06-01,1997-06,5.9375
1,1997-07-01,1997-07,5.9375
2,1997-08-01,1997-08,5.9375
3,1997-09-01,1997-09,5.9375
4,1997-10-01,1997-10,5.9375


In [7]:
merged_with_rate['Bank Rate'].isna().sum()


np.int64(0)

In [10]:
merged_with_rate.head()

,CPI ANNUAL RATE 00: ALL ITEMS 2015=100,Date,YearMonth,Gross Value Added - Monthly (3 month on 3 month growth) :CVM SA,Exchange_USD,"Unemployment rate (aged 16 and over, seasonally adjusted): %",RPI: Percentage change over 12 months - Petrol and Oil incl Fuel Oil,Bank Rate
0,1.7,1997-06-01,1997-06,1.0,1.6446,7.3,9.3,5.9375
1,2.0,1997-07-01,1997-07,0.6,1.6702,7.1,14.0,5.9375
2,2.0,1997-08-01,1997-08,0.9,1.6034,6.8,14.1,5.9375
3,1.8,1997-09-01,1997-09,0.8,1.6015,6.7,11.2,5.9375
4,1.9,1997-10-01,1997-10,1.0,1.6329,6.6,8.5,5.9375


In [11]:
merged_with_rate.tail()

,CPI ANNUAL RATE 00: ALL ITEMS 2015=100,Date,YearMonth,Gross Value Added - Monthly (3 month on 3 month growth) :CVM SA,Exchange_USD,"Unemployment rate (aged 16 and over, seasonally adjusted): %",RPI: Percentage change over 12 months - Petrol and Oil incl Fuel Oil,Bank Rate
333,2.6,2025-03-01,2025-03,0.7,1.2911,4.6,-5.4,4.75
334,3.5,2025-04-01,2025-04,0.7,1.3131,4.7,-9.8,4.75
335,3.4,2025-05-01,2025-05,0.5,1.3366,4.7,-11.5,4.75
336,3.6,2025-06-01,2025-06,0.3,1.3566,4.7,-10.0,4.75
337,3.8,2025-07-01,2025-07,0.2,1.3492,4.8,-7.0,4.75


In [12]:
merged_with_rate.to_csv("final_merged_dataset.csv", index=False)
print("Saved as final_merged_dataset.csv")


Saved as final_merged_dataset.csv
